In [1]:
import pandas as pd

df = pd.read_csv('data.csv')

In [2]:
df.columns

Index(['Date', 'Age ', 'BMI', 'Surgical Approach',
       'Pre-op visual impairment (y/n) - symptomatic complaint of visual loss',
       'Pre-op hydrocephalus (y/n)', 'Pre-op intacranial pressure symptoms',
       'Pre-op headache', 'Pre-op endocrine changes (y/n)',
       'Pre-op panhypopit (y/n) - define number of hormone axes that are non-functional',
       'Axes disrupted', 'Tumor size', 'Tumor size.1', 'Tumor size.2',
       'Tumor volume', 'Tumor (Adamantinomatous, papillary)', 'BRAF+',
       'Location (purely intrasellar, suprasellar/sellar, suprasellar/sellar with pseudoventricular extension, suprasellar/sellar with intraventricular extension, purely intraventricular)',
       'Intraventricular extension', 'Solid, cystic, mixed',
       'Resection outcome (total, subtotal, partial)', 'Prior operation',
       'recurring tumor?', 'Recurrence after this operation',
       'Recurrence date', 'Days to recurrence',
       'Re-operation after this operation (add date)', 'Time to r

In [3]:
# === EDA for CP Permanent DI paper ===
# Generates cohort summaries and 300 dpi figures into ./figs/
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Optional (nice styling). If seaborn not installed, comment it out.
import seaborn as sns
sns.set_context("talk")
sns.set_style("whitegrid")

# Optional missingness visualization
try:
    import missingno as msno
    HAS_MSNO = True
except Exception:
    HAS_MSNO = False

# -------------------------
# 0) Paths & constants
# -------------------------
DATA_PATH = "data.csv"
FIG_DIR = "figs"
os.makedirs(FIG_DIR, exist_ok=True)

# Column names used repeatedly (keep exact spellings)
COL_AGE = "Age "
COL_BMI = "BMI"
COL_LAST_BMI = "Last BMI on file"

COL_TS1 = "Tumor size"
COL_TS2 = "Tumor size.1"
COL_TS3 = "Tumor size.2"
COL_TVOL = "Tumor volume"

COL_VIS = "Pre-op visual impairment (y/n) - symptomatic complaint of visual loss"
COL_HCP = "Pre-op hydrocephalus (y/n)"
COL_ICP = "Pre-op intacranial pressure symptoms"
COL_HA  = "Pre-op headache"
COL_END = "Pre-op endocrine changes (y/n)"
COL_PAN = "Pre-op panhypopit (y/n) - define number of hormone axes that are non-functional"
COL_IVX = "Intraventricular extension"

COL_SAPP = "Surgical Approach"
COL_RO   = "Resection outcome (total, subtotal, partial)"

COL_SOLIDMIX = "Solid, cystic, mixed"
COL_LOC = "Location (purely intrasellar, suprasellar/sellar, suprasellar/sellar with pseudoventricular extension, suprasellar/sellar with intraventricular extension, purely intraventricular)"

COL_PERMDI = "Permanent DI"

# Convenience buckets
CONTINUOUS = [COL_AGE, COL_BMI, COL_LAST_BMI, COL_TS1, COL_TS2, COL_TS3, COL_TVOL]
BINARY = [COL_VIS, COL_HCP, COL_ICP, COL_HA, COL_END, COL_PAN, COL_IVX]
CATEGORICAL = [COL_SAPP, COL_RO, COL_SOLIDMIX, COL_LOC]

# -------------------------
# 1) Load & light cleaning
# -------------------------
df = pd.read_csv(DATA_PATH)

# Recompute tumor volume if missing or fully absent (ellipsoid)
if COL_TVOL not in df.columns:
    df[COL_TVOL] = np.nan

need = df[COL_TVOL].isna()
if all(c in df.columns for c in [COL_TS1, COL_TS2, COL_TS3]) and need.any():
    df.loc[need, COL_TVOL] = (4/3)*math.pi * (df.loc[need, COL_TS1]/2) * (df.loc[need, COL_TS2]/2) * (df.loc[need, COL_TS3]/2)

# Coerce numeric columns that might be object-typed
for c in [COL_AGE, COL_BMI, COL_LAST_BMI, COL_TS1, COL_TS2, COL_TS3, COL_TVOL]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# -------------------------
# 2) Cohort & outcome summary
# -------------------------
n = len(df)
n_with_outcome = df[COL_PERMDI].notna().sum()
prevalence = df[COL_PERMDI].mean() if n_with_outcome > 0 else np.nan

summary = pd.DataFrame({
    "N_total": [n],
    "N_outcome_nonmissing": [n_with_outcome],
    "Permanent_DI_prevalence": [prevalence]
})
summary.to_csv(os.path.join(FIG_DIR, "cohort_summary.csv"), index=False)
print(summary)

# Outcome prevalence figure
plt.figure(figsize=(4.5, 5))
vals = [df[COL_PERMDI].sum(), (df[COL_PERMDI]==0).sum()] if df[COL_PERMDI].notna().any() else [0,0]
labels = ["Permanent DI = 1", "Permanent DI = 0"]
plt.bar(labels, vals, color=["#4C78A8","#72B7B2"])
plt.ylabel("Count")
plt.title("Permanent DI Outcome Counts")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig_outcome_counts.png"), dpi=300)
plt.close()

# -------------------------
# 3) Missingness overview
# -------------------------
missing_counts = df.isna().sum().sort_values(ascending=False)
missing_counts.to_csv(os.path.join(FIG_DIR, "missing_counts.csv"))

plt.figure(figsize=(8, 10))
missing_counts.head(30).plot(kind="barh")
plt.gca().invert_yaxis()
plt.xlabel("Missing values")
plt.title("Missingness by feature (top 30)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig_missing_counts.png"), dpi=300)
plt.close()

if HAS_MSNO:
    # Matrix view
    plt.figure(figsize=(10, 6))
    msno.matrix(df)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "fig_missing_matrix.png"), dpi=300)
    plt.close()

    # Heatmap of missingness correlations
    plt.figure(figsize=(8, 6))
    msno.heatmap(df)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "fig_missing_heatmap.png"), dpi=300)
    plt.close()

# -------------------------
# 4) Continuous distributions
# -------------------------
def hist_kde(series, name):
    s = series.dropna()
    if len(s) == 0: 
        return
    plt.figure(figsize=(6, 4.2))
    sns.histplot(s, bins=15, kde=True, edgecolor="black")
    plt.xlabel(name)
    plt.ylabel("Frequency")
    plt.title(f"Distribution of {name}")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f"fig_dist_{name.replace(' ','_')}.png"), dpi=300)
    plt.close()

for col in [c for c in CONTINUOUS if c in df.columns]:
    hist_kde(df[col], col)

# Correlation (continuous only)
cont_available = [c for c in CONTINUOUS if c in df.columns]
if len(cont_available) >= 2:
    corr = df[cont_available].corr(method="spearman", min_periods=1)
    plt.figure(figsize=(6.8, 5.6))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", cbar_kws={"shrink": .8})
    plt.title("Spearman Correlation (Continuous Variables)")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "fig_corr_continuous.png"), dpi=300)
    plt.close()
    corr.to_csv(os.path.join(FIG_DIR, "corr_continuous.csv"))

# -------------------------
# 5) Categorical distributions
# -------------------------
def bar_counts(col):
    if col not in df.columns: 
        return
    vc = df[col].astype(str).str.strip().str.lower().value_counts(dropna=True)
    plt.figure(figsize=(7, 4.2))
    sns.barplot(x=vc.index, y=vc.values, color="#4C78A8")
    plt.ylabel("Count")
    plt.xlabel(col)
    plt.title(f"{col} (counts)")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    fname = f"fig_counts_{col[:30].replace(' ','_')}.png"
    plt.savefig(os.path.join(FIG_DIR, fname), dpi=300)
    plt.close()

for col in [c for c in CATEGORICAL if c in df.columns]:
    bar_counts(col)

# -------------------------
# 6) Outcome-stratified plots
# -------------------------
# 6a) Continuous vs outcome: box/violin
def cont_vs_outcome(col):
    if col not in df.columns or COL_PERMDI not in df.columns:
        return
    sub = df[[col, COL_PERMDI]].dropna()
    if sub[COL_PERMDI].nunique() != 2:
        return
    sub[COL_PERMDI] = sub[COL_PERMDI].astype(int)

    plt.figure(figsize=(5.2, 4.6))
    sns.boxplot(x=COL_PERMDI, y=col, data=sub)
    sns.stripplot(x=COL_PERMDI, y=col, data=sub, color="black", alpha=0.4, jitter=0.15)
    plt.xlabel("Permanent DI (0/1)")
    plt.ylabel(col)
    plt.title(f"{col} by Permanent DI")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f"fig_{col.replace(' ','_')}_by_DI_box.png"), dpi=300)
    plt.close()

    plt.figure(figsize=(5.2, 4.6))
    sns.violinplot(x=COL_PERMDI, y=col, data=sub, inner="box")
    plt.xlabel("Permanent DI (0/1)")
    plt.ylabel(col)
    plt.title(f"{col} by Permanent DI (Violin)")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f"fig_{col.replace(' ','_')}_by_DI_violin.png"), dpi=300)
    plt.close()

for col in [c for c in CONTINUOUS if c in df.columns]:
    cont_vs_outcome(col)

# 6b) Categorical vs outcome: stacked bars (proportion within category)
def cat_vs_outcome(col):
    if col not in df.columns or COL_PERMDI not in df.columns:
        return
    sub = df[[col, COL_PERMDI]].dropna()
    if sub.empty or sub[COL_PERMDI].nunique() != 2: 
        return

    # normalize categories
    sub[col] = sub[col].astype(str).str.strip().str.lower()
    tab = pd.crosstab(sub[col], sub[COL_PERMDI], normalize="index")  # proportions per category

    # stacked bar
    plt.figure(figsize=(7.0, 4.5))
    bottom = np.zeros(len(tab))
    for di_val in sorted(tab.columns):
        plt.bar(tab.index, tab[di_val].values, bottom=bottom, label=f"DI={di_val}")
        bottom += tab[di_val].values
    plt.ylabel("Proportion within category")
    plt.xlabel(col)
    plt.title(f"Proportion of Permanent DI by {col}")
    plt.xticks(rotation=25, ha="right")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f"fig_DI_by_{col[:30].replace(' ','_')}_stacked.png"), dpi=300)
    plt.close()

# Apply to key categorical predictors
for col in [c for c in [COL_SAPP, COL_RO, COL_SOLIDMIX, COL_LOC] if c in df.columns]:
    cat_vs_outcome(col)

# -------------------------
# 7) One-page summary table (overall + by outcome)
# -------------------------
def summarize_by_outcome(df, continuous, binary, categorical, outcome):
    rows = []

    # Overall and by outcome
    for label, d in [("Overall", df), ("DI=0", df[df[outcome]==0]), ("DI=1", df[df[outcome]==1])]:
        if d.empty: 
            continue

        # Continuous
        for c in [x for x in continuous if x in d.columns]:
            s = d[c].dropna()
            rows.append({
                "Group": label,
                "Feature": c,
                "Type": "continuous",
                "N": len(s),
                "Mean": np.round(s.mean(), 3) if len(s) else np.nan,
                "SD": np.round(s.std(ddof=1), 3) if len(s) else np.nan,
                "Median": np.round(s.median(), 3) if len(s) else np.nan,
                "IQR": np.round((s.quantile(0.75)-s.quantile(0.25)), 3) if len(s) else np.nan
            })

        # Binary/categorical as counts
        for c in [x for x in binary + categorical if x in d.columns]:
            vc = d[c].value_counts(dropna=True)
            # count of 1 for binary; otherwise top categories
            if c in binary:
                n1 = int((d[c]==1).sum())
                n0 = int((d[c]==0).sum())
                rows.append({"Group": label, "Feature": c, "Type": "binary", "N": int(d[c].notna().sum()), "Count_1": n1, "Count_0": n0})
            else:
                # categorical
                topk = vc.head(5)
                desc = "; ".join([f"{idx}:{cnt}" for idx, cnt in topk.items()])
                rows.append({"Group": label, "Feature": c, "Type": "categorical", "N": int(d[c].notna().sum()), "Top5": desc})

    return pd.DataFrame(rows)

summary_tbl = summarize_by_outcome(df, CONTINUOUS, BINARY, CATEGORICAL, COL_PERMDI)
summary_tbl.to_csv(os.path.join(FIG_DIR, "summary_by_outcome.csv"), index=False)
print("Saved figures →", FIG_DIR)


   N_total  N_outcome_nonmissing  Permanent_DI_prevalence
0       59                    58                 0.655172


C:\Users\shrin\AppData\Local\Temp\ipykernel_34752\2446678096.py:114: UserWarning: Tight layout not applied. The left and right margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
C:\Users\shrin\AppData\Local\Temp\ipykernel_34752\2446678096.py:122: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
C:\Users\shrin\AppData\Local\Temp\ipykernel_34752\2446678096.py:129: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


Saved figures → figs


<Figure size 1000x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>